# 08 - Fusion: the regime-aware trust score

**GPU preferred (embedding extraction); Run all; resumes** via saved predictions.

Population: the probe universe (~50k), where every domain was actually probed
so certificate absence is an observation. Two frozen splits, three seeds.

Component decisions carried in from 05-07a:

* **DGA regime -> handcrafted lexical without `tld`** (`xgb_tierA_notld`:
  best family-disjoint FPR@95%TPR, stable). `tld` is a family fingerprint.
* **Character model -> ablation row only.** Both CNN variants memorise
  families; the learned embedding enters as row (d) to measure whether it adds
  anything *inside* a regularised fused model.
* **Certificate regime -> presence + properties** (tier B result, family-invariant).

Fusion rows, all under one protocol:

| row | features | role |
|---|---|---|
| (a) lexical | lexical, no tld | DGA-regime score alone |
| (b) cert | presence + properties | certificate-regime score alone |
| (c) **fused** | (a) + (b), one XGBoost | **primary trust score** |
| (d) fused+emb | (c) + 64-d CNN embedding | learned-representation ablation |
| (e) late | mean of calibrated (a) and (b) | simple-fusion control |

**Regime breakdown.** Every row is additionally evaluated on the two test
subpopulations - domains without a certificate (the DGA regime) and
certificate-holders (the live-infrastructure regime, ~1.2% prevalence). A
single score, reported per regime: that is what "adaptive" means empirically.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard xgboost

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb, torch, yaml, inspect
from pathlib import Path
from src.evaluate import splits, metrics, predictions
from src.utils import manifest as mf
from src.models.cnn_bilstm import CharEncoder, CNNBiLSTM
from src.models.calibrate import Calibrator

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
XDEV = 'cuda' if DEV == 'cuda' else 'cpu'
print('device:', DEV)

X_all = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
probe = set(pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")['domain'])
B = X_all[X_all['domain'].isin(probe)].copy().reset_index(drop=True)
print('probe universe:', B.shape, f'| malicious {B.label.mean():.4f} | cert-holders {int(B.has_certificate.sum())}')

PRED_DIR, MODEL_DIR = Path(P['artifacts']['predictions']), Path(P['artifacts']['models'])
LEXICAL = ['length','core_length','n_labels','shannon_entropy','vowel_ratio',
           'digit_ratio','hyphen_count','max_consec_consonants','bigram_score',
           'trigram_score','unique_char_ratio','is_idn','has_digit','starts_with_digit']
CERT_NUM = ['has_certificate','is_free_ca','validity_days','days_until_expiry',
            'cert_age_days','is_expired','is_not_yet_valid','is_self_signed',
            'san_count','wildcard_san','cn_in_san','key_bits','short_validity',
            'very_fresh_cert','cn_san_mismatch','weak_key']
CERT_CATS = ['issuer_org','key_algorithm','sig_algorithm','san_bucket']

def train_categories(df, cats): return {c: pd.Index(df[c].dropna().unique()) for c in cats}
def matrix(df, num, cats, categories):
    Xm = df[num].apply(pd.to_numeric, errors='coerce').astype(np.float32)
    for c in cats:
        known = df[c].where(df[c].isin(categories[c]))
        Xm[c] = pd.Categorical(known, categories=categories[c])
    return Xm

XGB = dict(objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
           device=XDEV, enable_categorical=True, max_depth=6, learning_rate=0.05,
           n_estimators=2000, early_stopping_rounds=100, subsample=0.8,
           colsample_bytree=0.8, min_child_weight=5, reg_lambda=2.0, verbosity=0)

## CNN embeddings (split- and seed-matched)

The embedding for a given split and seed comes from the CNN trained on **that
split's training part** in notebook 07. Using any other model would leak
test-family information into the fused features.

In [ ]:
cfg = yaml.safe_load(open(f'{REPO}/configs/cnn_bilstm.yaml'))
MODEL_KW = {k: v for k, v in cfg['model'].items()
            if k in inspect.signature(CNNBiLSTM.__init__).parameters}
enc = CharEncoder(cfg['input']['charset'], cfg['input']['max_length'])
EMB_DIR = Path(P['data']['features'])/'embeddings'; EMB_DIR.mkdir(exist_ok=True)

@torch.no_grad()
def embed(split_name, seed):
    out = EMB_DIR/f'cnn_emb_probe_{split_name}_s{seed}.parquet'
    if out.exists():
        return pd.read_parquet(out)
    w = MODEL_DIR/f'cnnbilstm_tierA_{split_name}_s{seed}.pt'
    net = CNNBiLSTM(enc.vocab_size, **MODEL_KW).to(DEV)
    net.load_state_dict(torch.load(w, map_location=DEV)); net.eval()
    embs = []
    for i in range(0, len(B), 4096):
        x = torch.from_numpy(enc.encode_batch(B['domain'].values[i:i+4096])).to(DEV)
        embs.append(net.embed(x).cpu().numpy())
    E = pd.DataFrame(np.vstack(embs), columns=[f'emb_{i}' for i in range(MODEL_KW['penultimate_dim'])])
    E.insert(0, 'domain', B['domain'].values)
    E.to_parquet(out, index=False); return E

for sp_ in ['random_v1','family_disjoint_v1']:
    for sd in (42,43,44):
        E = embed(sp_, sd); print(sp_, sd, E.shape)

## Fusion grid

In [ ]:
ROWS = {
    'a_lexical':  (LEXICAL, [], False),
    'b_cert':     (CERT_NUM, CERT_CATS, False),
    'c_fused':    (LEXICAL + CERT_NUM, CERT_CATS, False),
    'd_fused_emb':(LEXICAL + CERT_NUM, CERT_CATS, True),
}
SPLITS, SEEDS = ['family_disjoint_v1','random_v1'], [42,43,44]

def regime_metrics(te, scores):
    """Overall plus per-regime metrics from one prediction vector."""
    out = metrics.evaluate(te['label'].values, scores)
    for name, mask in [('nocert', ~te['has_certificate'].astype(bool)),
                       ('cert',    te['has_certificate'].astype(bool))]:
        sub, sc = te[mask.values], scores[mask.values]
        if sub['label'].nunique() == 2:
            m = metrics.evaluate(sub['label'].values, sc)
            out.update({f'{name}_roc_auc': m['roc_auc'], f'{name}_pr_auc': m['pr_auc'],
                        f'{name}_fpr_at_95_tpr': m['fpr_at_95_tpr'],
                        f'{name}_positive_rate': m['positive_rate'], f'{name}_n': m['n']})
    return out

for split_name in SPLITS:
    sp = splits.load_split(P['data']['splits'], split_name); d = sp['domains']
    parts = {k: B[B['domain'].isin(d[k])].reset_index(drop=True) for k in ('train','val','test')}
    for seed in SEEDS:
        E = embed(split_name, seed)
        for row, (num, cats, use_emb) in ROWS.items():
            run_id = f'fusion_{row}_{split_name}_s{seed}'
            if (PRED_DIR/f'{run_id}.parquet').exists():
                print('SKIP (done)', run_id); continue
            fr = {k: v.merge(E, on='domain', how='left') if use_emb else v for k, v in parts.items()}
            num_ = num + ([c for c in E.columns if c.startswith('emb_')] if use_emb else [])
            categories = train_categories(fr['train'], cats)
            Xtr = matrix(fr['train'], num_, cats, categories); ytr = fr['train']['label'].values
            Xva = matrix(fr['val'],   num_, cats, categories); yva = fr['val']['label'].values
            Xte = matrix(fr['test'],  num_, cats, categories)
            spw = float((ytr==0).sum()) / max((ytr==1).sum(), 1)
            model = xgb.XGBClassifier(scale_pos_weight=spw, random_state=seed, **XGB)
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            scores = model.predict_proba(Xte)[:,1]
            val_scores = model.predict_proba(Xva)[:,1]
            m = regime_metrics(fr['test'], scores); m['best_iteration'] = int(model.best_iteration)
            predictions.save(run_id, PRED_DIR, fr['test']['domain'].values, fr['test']['label'].values,
                             scores, extra={'has_certificate': fr['test']['has_certificate'].values})
            predictions.save(run_id+'_VAL', PRED_DIR, fr['val']['domain'].values, yva, val_scores)
            model.save_model(str(MODEL_DIR/f'{run_id}.json'))
            mf.record(P['manifest'], run_id, f'fusion_{row}', {'num': num_, 'cat': cats,
                      'use_embedding': use_emb, 'xgb': XGB}, split_name, sp['split_file'],
                      m, seed, repo_root=REPO)
            print(f'DONE {run_id:44s} roc={m["roc_auc"]:.4f} fpr95={m["fpr_at_95_tpr"]:.4f} | '
                  f'nocert roc={m.get("nocert_roc_auc",float("nan")):.4f} | '
                  f'cert roc={m.get("cert_roc_auc",float("nan")):.4f} pr={m.get("cert_pr_auc",float("nan")):.4f}')
print('grid complete')

## Row (e) - late fusion control

Average of the *calibrated* (a) and (b) scores. Calibrated on the validation
part (isotonic), so both components are on a probability scale before
averaging. If (c) does not beat (e), a single joint model was unnecessary.

In [ ]:
for split_name in SPLITS:
    sp = splits.load_split(P['data']['splits'], split_name)
    for seed in SEEDS:
        run_id = f'fusion_e_late_{split_name}_s{seed}'
        if (PRED_DIR/f'{run_id}.parquet').exists():
            print('SKIP (done)', run_id); continue
        comps = []
        for row in ('a_lexical','b_cert'):
            te = predictions.load(f'fusion_{row}_{split_name}_s{seed}', PRED_DIR)
            va = predictions.load(f'fusion_{row}_{split_name}_s{seed}_VAL', PRED_DIR)
            cal = Calibrator('isotonic').fit(va['raw_score'], va['true_label'])
            comps.append(te.assign(cal=cal.transform(te['raw_score']))[['domain','true_label','has_certificate','cal']])
        j = comps[0].merge(comps[1][['domain','cal']], on='domain', suffixes=('_a','_b'))
        scores = ((j['cal_a'] + j['cal_b'])/2).values
        te_like = j.rename(columns={'true_label':'label'})
        m = regime_metrics(te_like, scores)
        predictions.save(run_id, PRED_DIR, j['domain'].values, j['true_label'].values, scores,
                         extra={'has_certificate': j['has_certificate'].values})
        mf.record(P['manifest'], run_id, 'fusion_e_late', {'method': 'mean of isotonic-calibrated a,b'},
                  split_name, sp['split_file'], m, seed, repo_root=REPO)
        print(f'DONE {run_id:44s} roc={m["roc_auc"]:.4f} fpr95={m["fpr_at_95_tpr"]:.4f}')

## Results

In [ ]:
man = mf.load_manifest(P['manifest'])
f = man[man['run_family'].str.startswith('fusion_')].copy()
f['row'] = f['run_family'].str.replace('fusion_','')

cols = ['metrics.roc_auc','metrics.fpr_at_95_tpr','metrics.pr_auc']
overall = f.groupby(['row','split_name'])[cols].agg(['mean','std']).round(4)
print('OVERALL (probe universe)'); display(overall)

reg = f.groupby(['row','split_name'])[
    ['metrics.nocert_roc_auc','metrics.nocert_fpr_at_95_tpr',
     'metrics.cert_roc_auc','metrics.cert_pr_auc','metrics.cert_fpr_at_95_tpr']].mean().round(4)
print('PER REGIME (mean over seeds)'); display(reg)

overall.to_csv(Path(P['results']['tables'])/'table_fusion_overall.csv')
reg.to_csv(Path(P['results']['tables'])/'table_fusion_per_regime.csv')

In [ ]:
# The three comparisons the paper makes from this table
piv = f.groupby(['row','split_name'])['metrics.fpr_at_95_tpr'].mean().unstack('split_name').round(4)
print('FPR@95%TPR by row (lower is better)'); display(piv)
fd = piv['family_disjoint_v1']
print()
print(f'fusion gain over lexical alone   (c vs a): {fd["a_lexical"]-fd["c_fused"]:+.4f}')
print(f'fusion gain over cert alone      (c vs b): {fd["b_cert"]-fd["c_fused"]:+.4f}')
print(f'joint model vs late fusion       (c vs e): {fd["e_late"]-fd["c_fused"]:+.4f}')
print(f'embedding adds inside fusion     (d vs c): {fd["c_fused"]-fd["d_fused_emb"]:+.4f}')

---

**The primary trust score is row (c)**, `fusion_c_fused`, unless (d) beats it
family-disjoint with acceptable variance. Notebooks 09 (calibration) and 10
(SHAP, explanation quality) operate on that run family.

The per-regime table is the paper's central result: one score, its behaviour
in the DGA regime and in the live-infrastructure regime reported separately,
with the honest prevalence of each.